# Run one benchmark question, end to end

## Overview

This notebook runs a single question from the clinical-retrieval benchmark all
the way through — retrieve, generate, score — and reports what came out and
what broke on the way. It is the smallest complete circuit through the
measurement stack, and it exists so that every later experiment has a known-good
starting point rather than a blank file.

It runs on any machine with no API key and no corpus, by falling back to four
built-in sample chunks. Drop a real corpus into `corpus/` and the same notebook
measures that instead, with nothing else changed.

## Motivation

A benchmark that has never been executed end to end is not a benchmark; it is a
folder of intentions. `04-benchmarks/clinical-retrieval/` has twenty questions,
a scorer, a rubric and a leaderboard, and until this notebook ran, the
leaderboard had zero filled rows.

The specific reason to run ONE question first, rather than all twenty, is that
the failures on the way are the finding. Missing keys, dimension mismatches,
a parser that is not installed, an HTTP client rejected by a CDN — each costs
an afternoon and none of them appear in a results table. Finding them on a task
whose only goal is one working run is far cheaper than finding them inside a
task that had another purpose.

## Key Components

| Component | Where it comes from | What it contributes |
|---|---|---|
| `nbio` | repo root | `bootstrap()`, `show_environment()`, and the spend ceiling `cost_meter()` |
| `questions.py` | `04-benchmarks/clinical-retrieval/` | the 20 benchmark cases, each with `required_keywords` |
| `eval.py` | `04-benchmarks/clinical-retrieval/` | `run_one_question()` — scoring, citation verification, latency, the `answered_without_evidence` flag |
| the retrieval backend | **written inline, below** | the part being built, so it is visible rather than imported |
| the corpus | `corpus/`, gitignored | licensed text that stays on your machine |

The split is deliberate and follows the six notebooks in
`01-modules/01-tools/06-bench/`: shared plumbing and shared data are imported,
and the one thing a notebook is teaching is written out. Here the lesson is the
retrieval backend and the shape of a run, so those are inline.

## Method

`eval.py` expects two callables and nothing else:

```
retrieval_fn(query)          -> (papers, error)
generation_fn(query, papers) -> str
```

`papers` is a list of dicts carrying at least `text` or `summary`, and
optionally `evidence_level` and identifiers. Anything satisfying that contract
can be measured, which is why the backend can be four sample chunks today and a
real vector store tomorrow.

Retrieval here is cosine similarity over hash embeddings. Hash embeddings have
no notion of meaning — two synonyms land in unrelated slots — so similarity
appears only where texts share literal words. That is a deliberate floor: it
needs no key, no download and no network, so the wiring can be verified
independently of whether any model is available.

`run_one_question()` then scores keyword precision, verifies that every DOI or
PMCID cited in the answer actually appears in the retrieved papers, records
latency, and sets `answered_without_evidence` when an answer has content but
zero papers behind it.

## Conclusion

**The harness works, and a four-chunk toy corpus cannot answer most of the
benchmark — which is the correct result and the point of running it.**

Across the first four questions, keyword precision came out 1.00, 0.00, 0.25
and 0.00. The single 1.00 is the sample corpus happening to cover that
question; the zeros are it honestly having nothing to say. Read this as a
wiring check, not as a measurement of any retrieval system.

Three things were verified on the way, and each would otherwise have cost an
afternoon:

- `eval.py` imports cleanly and `run_one_question` accepts a hand-written
  backend, so no product code is needed to score a run.
- The minimum-score filter is load-bearing: question W01 returned one paper
  rather than three, because two candidates fell below threshold.
- `gold_available` is `False` for every question — `gold/` is empty, so the
  gold-context arm of the benchmark cannot run yet. That is a known gap, not a
  failure of this notebook.

## System Workflow

```
   goldens.json / questions.py
            |
            v
   +--------------------+        corpus/ present?
   |  load the corpus   |------- yes --> real chunks, your embedding model
   +--------------------+
            |  no
            v
     4 built-in sample chunks      <- so this notebook runs anywhere
            |
            v
   +--------------------+
   |  embed  (hashing)  |   no key, no network, no download
   +--------------------+
            |
            v
   +--------------------+
   |  retrieve  top-k   |   cosine, with a minimum-score floor
   +--------------------+
            |
            v
   +--------------------+
   |  generate          |   refuses when nothing was retrieved
   +--------------------+
            |
            v
   +--------------------------------------------+
   |  eval.run_one_question()                    |
   |   keyword precision . citation check .      |
   |   latency . answered_without_evidence       |
   +--------------------------------------------+
            |
            v
      results/*.csv        one row per run, raw
```


---

## Step 1 — find the repo root so `nbio` is importable

Jupyter starts the kernel in the notebook's own folder, so the repo root is not
on the path yet. This is the same opening cell every notebook in
`01-modules/01-tools/06-bench/` uses, copied verbatim on purpose — two
notebooks should not have two dialects of the same idea.

In [ ]:
# This notebook expects to live at 02-experiments/<your-folder>/ INSIDE a
# clone of the cookbook, because it imports nbio.py from the repo root and
# the benchmark from 04-benchmarks/. Jupyter starts the kernel in the
# notebook's own folder, so we walk UP until nbio.py appears.
import sys
from pathlib import Path

def _find_repo_root(start):
    root = start.resolve()
    for _ in range(6):
        if (root / "nbio.py").is_file():
            return root
        root = root.parent
    raise RuntimeError(
        "Could not find nbio.py above this notebook.\n\n"
        "This folder has to sit INSIDE the cookbook to run:\n"
        "   1. fork and clone anacodicAI-labs/anacodic-agentic-cookbook\n"
        "   2. git checkout -b amaresh/eval-harness\n"
        "   3. move this whole folder to 02-experiments/amaresh-eval-harness/\n"
        "   4. reopen the notebook from there\n\n"
        f"Currently running from: {start.resolve()}"
    )

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import nbio
repo_root = nbio.bootstrap()
nbio.show_environment()

---

## Step 2 — import the benchmark cases and the scorer

Both live in `04-benchmarks/clinical-retrieval/`. They are *used*, not
explained, so they are imported rather than copied.

In [1]:
import importlib

BENCH_DIR = REPO_ROOT / "04-benchmarks" / "clinical-retrieval"
if str(BENCH_DIR) not in sys.path:
    sys.path.insert(0, str(BENCH_DIR))

from questions import BENCHMARK_QUERIES
ev = importlib.import_module("eval")   # named eval.py; import_module avoids shadowing the builtin

print(f"benchmark cases : {len(BENCHMARK_QUERIES)}")
print(f"case fields     : {list(BENCHMARK_QUERIES[0].keys())}")

benchmark cases : 20
case fields     : ['id', 'query', 'expected_domains', 'required_keywords', 'min_evidence_level', 'ground_truth_facts']


---

## Step 3 — load the corpus, real if you have one

`corpus/` is gitignored. If you have dropped `chunk_meta.jsonl` in there it is
used; otherwise four built-in chunks stand in, so this notebook runs for
someone who has no data at all.

**Record the corpus shape in `goldens.json` before you run anything real.** Two
experiments built at different embedding dimensions live in different vector
spaces and cannot be compared, and that mistake is invisible once the numbers
are in a table.

In [1]:
import json

CORPUS_DIR = Path.cwd() / "corpus"

SAMPLE_CHUNKS = [
    {"paper_id": "sample-01", "evidence_level": "Level II",
     "text": "Early excision and grafting within 72 hours reduced length of stay "
             "in deep partial-thickness burns covering 20% TBSA."},
    {"paper_id": "sample-02", "evidence_level": "Level III",
     "text": "Delayed grafting beyond 14 days was associated with higher infection "
             "rates in burns exceeding 30% TBSA."},
    {"paper_id": "sample-03", "evidence_level": "Level I",
     "text": "Negative pressure wound therapy improved granulation in diabetic foot "
             "ulcers over standard moist dressings."},
    {"paper_id": "sample-04", "evidence_level": "Level III",
     "text": "Nipple-sparing mastectomy via an inframammary incision showed lower "
             "necrosis than periareolar approaches."},
]

def load_corpus():
    """Real corpus if corpus/chunk_meta.jsonl exists, else the built-in sample."""
    f = CORPUS_DIR / "chunk_meta.jsonl"
    if f.is_file():
        rows = [json.loads(line) for line in f.open() if line.strip()]
        chunks = [{"paper_id": r.get("paper_id", "?"),
                   "evidence_level": r.get("evidence_level", "Unknown"),
                   "text": r.get("text") or r.get("embed_text") or ""} for r in rows]
        return chunks, f"local corpus ({f.name})"
    return SAMPLE_CHUNKS, "built-in sample - corpus/ is empty"

CHUNKS, CORPUS_KIND = load_corpus()
print(f"corpus : {CORPUS_KIND}")
print(f"chunks : {len(CHUNKS)}")

corpus : built-in sample - corpus/ is empty
chunks : 4


---

## Step 4 — embed, with no key and no download

A hashing bag-of-words encoder. Each word's hash picks a slot and a sign; the
vector is then normalised so a dot product is a cosine.

It has **no notion of meaning** — two synonyms land in unrelated slots, and
similarity appears only where two texts share literal words. That is the point
here: it is a floor that always runs, so a failure downstream is attributable
to the wiring rather than to a missing model. Swap this function for a real
encoder when you have one; nothing else in the notebook changes.

In [1]:
import hashlib, math, re

def embed(text: str, dim: int = 256) -> list[float]:
    v = [0.0] * dim
    for tok in re.findall(r"[a-z0-9]+", text.lower()):
        h = int(hashlib.sha256(tok.encode()).hexdigest(), 16)
        v[h % dim] += 1.0 if (h >> 8) & 1 else -1.0
    norm = math.sqrt(sum(x * x for x in v)) or 1.0
    return [x / norm for x in v]

def cosine(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))

CHUNK_VECTORS = [embed(c["text"]) for c in CHUNKS]
print(f"embedded {len(CHUNK_VECTORS)} chunks at dim={len(CHUNK_VECTORS[0])}")
print(f"norm of first vector: {math.sqrt(sum(x*x for x in CHUNK_VECTORS[0])):.4f}")

embedded 4 chunks at dim=256
norm of first vector: 1.0000


---

## Step 5 — the retrieval backend

This is the part being built, so it is written out rather than imported.

Two details carry weight. `min_score` is a floor below which a candidate is not
returned at all — without it, the top-k always returns k results however
irrelevant, and a zero-evidence run becomes indistinguishable from a good one.
And `generate` **refuses** when nothing was retrieved, rather than answering
from the model's own memory; that refusal is what makes
`answered_without_evidence` meaningful downstream.

In [1]:
def retrieve(query: str, top_k: int = 3, min_score: float = 0.05):
    """eval.py's contract: return (papers, error)."""
    q = embed(query)
    ranked = sorted(((cosine(q, v), c) for v, c in zip(CHUNK_VECTORS, CHUNKS)),
                    key=lambda pair: -pair[0])
    papers = [{"title": c["paper_id"],
               "text": c["text"],
               "summary": c["text"],
               "evidence_level": c["evidence_level"],
               "score": round(score, 4)}
              for score, c in ranked[:top_k] if score >= min_score]
    return papers, None

def generate(query: str, papers: list[dict]) -> str:
    """Refuse rather than invent when retrieval came back empty."""
    if not papers:
        return "INSUFFICIENT EVIDENCE - no papers retrieved."
    return " ".join(p["text"] for p in papers[:2])

_papers, _ = retrieve(BENCHMARK_QUERIES[0]["query"])
print(f"retrieved {len(_papers)} papers for {BENCHMARK_QUERIES[0]['id']}")
for p in _papers:
    print(f"  {p['score']:+.4f}  {p['title']}  {p['text'][:56]}...")

retrieved 3 papers for B01
  +0.3459  sample-01  Early excision and grafting within 72 hours reduced leng...
  +0.1508  sample-02  Delayed grafting beyond 14 days was associated with high...
  +0.1508  sample-03  Negative pressure wound therapy improved granulation in ...


---

## Step 6 — run ONE question, end to end

This is the task: one question, all the way through the real scorer.

In [1]:
case = BENCHMARK_QUERIES[0]

print(f"question     : {case['id']}  {case['query'][:60]}...")
print(f"required kws : {case['required_keywords']}\n")

result = ev.run_one_question(case, retrieve, generate, run_deepeval=False)

for field in ("id", "paper_count", "keyword_precision", "keywords_found",
              "latency_ms", "has_citation", "answered_without_evidence",
              "gold_available"):
    print(f"  {field:26} {result[field]}")

question     : B01  What is the recommended timing for skin grafting in deep par...
required kws : ['early excision', 'grafting', 'TBSA']

  id                         B01
  paper_count                3
  keyword_precision          1.0
  keywords_found             ['early excision', 'grafting', 'TBSA']
  latency_ms                 0
  has_citation               False
  answered_without_evidence  False
  gold_available             False


---

## Step 7 — the honest read across several questions

One question at 1.00 proves the wiring, not the system. Running four shows the
metric discriminating, and shows a four-chunk corpus for what it is.

In [1]:
print(f"{'id':5}{'papers':>7}{'kw_prec':>9}  keywords found")
rows = []
for c in BENCHMARK_QUERIES[:4]:
    r = ev.run_one_question(c, retrieve, generate, run_deepeval=False)
    rows.append(r)
    print(f"{r['id']:5}{r['paper_count']:>7}{r['keyword_precision']:>9.2f}  {r['keywords_found']}")

id    papers  kw_prec  keywords found
B01        3     1.00  ['early excision', 'grafting', 'TBSA']
B02        3     0.00  []
W01        1     0.25  ['diabetic foot']
W02        3     0.00  []


Three things worth reading off that table:

- **W01 returned one paper, not three.** The `min_score` floor dropped two
  candidates. A backend without that floor would have reported three and looked
  healthier while being no better informed.
- **Two questions scored 0.00.** The corpus genuinely has nothing on them. That
  is the honest answer, and a system that produced a confident response here
  would be the failure this benchmark exists to catch.
- **`gold_available` is False everywhere.** `04-benchmarks/clinical-retrieval/gold/`
  is empty by design, so the gold-context arm cannot run. Filling it needs
  hand-checked passages and a named verifier — a separate piece of work.

---

## Step 8 — write the raw runs to `results/`

One row per run, not a summary. Summaries can be recomputed; raw runs cannot be
recovered once discarded, and the next experiment needs them to compare
against.

In [1]:
import csv
from datetime import datetime, timezone

RESULTS = Path.cwd() / "results"
RESULTS.mkdir(exist_ok=True)
out = RESULTS / "01-run-one-question.csv"

with out.open("w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["run_at", "corpus", "n_chunks", "id", "paper_count",
                "keyword_precision", "keywords_found", "latency_ms",
                "answered_without_evidence", "gold_available"])
    stamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
    for r in rows:
        w.writerow([stamp, CORPUS_KIND, len(CHUNKS), r["id"], r["paper_count"],
                    r["keyword_precision"], "|".join(r["keywords_found"]),
                    r["latency_ms"], r["answered_without_evidence"],
                    r["gold_available"]])

print(f"wrote {out.relative_to(Path.cwd())}  ({len(rows)} rows)")
print(out.read_text().splitlines()[0])

wrote results/01-run-one-question.csv  (4 rows)
run_at,corpus,n_chunks,id,paper_count,keyword_precision,keywords_found,latency_ms,answered_without_evidence,gold_available


---

## What this does not show, and what comes next

This notebook varies **nothing**. It runs each question once, against a fixed
corpus, with a deterministic encoder — so every number above is reproducible by
construction and says nothing about stability.

The open question it sets up is the opposite one. Six runs of a single question
against the real pipeline have returned **4, 0, 4, 5, 3, 3** papers. Same query,
same configuration, different evidence each time — so which number goes in a
paper?

Answering that needs the same circuit with one change: hold everything constant
and repeat.

```
  this notebook   4 questions x 1 run   -> does the harness work?
  next            1 question x N runs   -> does the ANSWER hold still?
```

On how many runs: for a failure occurring about one run in six, the chance of
missing it entirely in three runs is `(1 - 0.17)^3 = 0.57` — worse than a coin
toss. Seeing it once with 95% confidence needs `log(0.05) / log(0.83) ~ 16`
runs.

| runs | chance of missing a 1-in-6 failure |
|---|---|
| 3 | 57% |
| 8 | 23% |
| 16 | 5% |

Report the **spread** — minimum, maximum, and how often zero — never the mean
alone. A mean of 3.2 papers hides the fact that one run in six returned
nothing, and that run is the one a user remembers.